# E0 — Baseline: Comparativa entre modelos

## Pregunta de investigación

¿Qué modelo es más eficiente en RPi5 en condiciones estándar?

## Hipótesis previa

Se espera encontrar diferencias significativas de throughput y eficiencia entre modelos, con mejora con ventilador activo.

## Configuración del experimento

| Parámetro | Valores |
|-----------|---------|
| Fan | Sí (1 run) / No (1 run) |
| Modelos | 6 modelos x 2 motores |
| Total | 24 runs |

In [1]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from monitorviz.io import load_collection
from monitorviz.transforms.collection import RunCollection
from monitorviz.viz import setup_style

setup_style("talk")
warnings.filterwarnings("ignore", category=UserWarning, module="seaborn")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# --- Carga de datos (E0-FAN + E0-NOFAN) ---
_here = Path.cwd()
PROJECT_ROOT = _here.parent if _here.name == "notebooks" else _here
DATA_ROOT = PROJECT_ROOT / "data" / "tfg-data"

e0_dirs = [DATA_ROOT / "E0-FAN", DATA_ROOT / "E0-NOFAN"]
for d in e0_dirs:
    assert d.is_dir(), f"No encontrado: {d.resolve()}"

runs = sum([load_collection(d).runs for d in e0_dirs], [])
coll = RunCollection(runs=runs)
print(f"Runs cargados E0: {len(coll)}")

# --- Filtrado para E0 ---
# E0: baseline sin acelerador, todos los modelos, ambos engines
summary = coll.summary_df()
hw_full = coll.hw_metrics_df()
pm_full = coll.prompt_metrics_df()

ex = summary[
    (summary["test_type"] == "TYPE_1") &
    (summary["accelerator"] == False)
].copy()

print(f"\nRuns en E0: {ex.shape[0]}")
print(f"Modelos únicos: {ex['model_label'].nunique()}")
print(f"Engines: {sorted(ex['engine'].unique())}")

Runs cargados E0: 36

Runs en E0: 36
Modelos únicos: 12
Engines: ['LLAMA', 'OLLAMA']


## Resumen ejecutivo

> TODO: Actualizar con resultados finales.

Placeholder: 3-5 frases resumiendo los hallazgos principales de E0.

## Comparativa de métricas de inferencia

Throughput, latencia, TTFT, perplejidad por configuración.

In [2]:
# Tabla resumen - Métricas de inferencia
display_cols = [
    "model_label", "engine",
    "tokens_per_s_mean", "words_per_s_mean",
    "latency_ms_mean", "ttft_ms_mean",
    "perplexity_geomean",
]
cols = [c for c in display_cols if c in ex.columns]
display(ex[cols].round(3))

,model_label,engine,tokens_per_s_mean,words_per_s_mean,latency_ms_mean,ttft_ms_mean,perplexity_geomean
0,ministral (O),OLLAMA,2.100,1.473,424964.275,20014.089,2.103
1,ministral (L),LLAMA,2.858,2.010,407140.417,3629.400,2.810
2,Llama-3.2-3B (O),OLLAMA,2.554,1.861,163999.251,12763.883,1.430
3,Llama-3.2-3B (L),LLAMA,2.384,1.740,850692.187,4291.243,2.286
4,Llama-3.2-1B (O),OLLAMA,6.830,5.137,60048.807,4603.280,1.654
5,Llama-3.2-1B (L),LLAMA,6.861,5.291,350077.805,1306.107,2.360
6,granite (O),OLLAMA,2.408,1.882,161774.734,20424.302,1.694
7,granite (L),LLAMA,2.781,2.148,143151.125,4423.324,2.854
8,gemma (O),OLLAMA,2.910,2.034,201334.289,12508.432,1.132
9,gemma (L),LLAMA,3.094,2.162,186211.320,5226.373,2.420


## Comparativa de hardware

Temperatura, frecuencia, CPU%, potencia, throttling.

In [3]:
# Tabla resumen - Hardware
if "temp_max_c" in ex.columns:
    hw_cols = [
        "model_label",
        "temp_max_c", "temp_mean_c",
        "power_mean_w", "power_max_w",
        "throttled_ratio",
    ]
    hw_cols = [c for c in hw_cols if c in ex.columns]
    display(ex[hw_cols].round(2))
else:
    print("Sin datos de hardware")

,model_label,temp_max_c,temp_mean_c,power_mean_w,power_max_w,throttled_ratio
0,ministral (O),96.35,92.30,5.15,9.01,0.99
1,ministral (L),95.80,91.47,4.87,9.68,1.00
2,Llama-3.2-3B (O),96.35,93.92,5.03,8.61,0.99
3,Llama-3.2-3B (L),96.35,92.95,4.85,9.57,1.00
4,Llama-3.2-1B (O),96.35,92.77,4.93,8.87,0.99
5,Llama-3.2-1B (L),95.80,93.05,4.78,9.10,1.00
6,granite (O),96.35,93.89,5.01,8.89,0.99
7,granite (L),95.80,93.35,4.94,9.61,1.00
8,gemma (O),96.35,93.58,4.95,8.86,0.99
9,gemma (L),95.80,93.03,4.86,9.36,1.00


## Timelines de temperatura y potencia por modelo

Small multiples con un subplot por modelo (2 columnas).

In [ ]:
# Timelines de temperatura
hw_ex = hw_full[hw_full["run_id"].isin(ex["run_id"])]

if not hw_ex.empty:
    models = sorted(ex["model_label"].unique())
    ncols = 2
    nrows = (len(models) + 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows))
    if len(models) == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, model in zip(axes, models):
        hw_m = hw_ex[hw_ex["model_label"] == model]
        if hw_m.empty:
            ax.text(0.5, 0.5, f"{model}\nSin datos", ha="center", va="center",
                   transform=ax.transAxes)
            ax.set_axis_off()
            continue

        sns.lineplot(data=hw_m, x="t_rel_s", y="temperature_c", ax=ax, alpha=0.7)
        ax.axhline(80, ls="--", color="red", alpha=0.5, label="80°C (throttle)")
        total_s = hw_m["t_rel_s"].max()
        ax.set_xlabel("Tiempo (min)" if total_s > 1800 else "Tiempo (s)")
        ax.set_ylabel("Temperatura (°C)")
        ax.set_title(f"{model}", fontweight="bold")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    for ax in axes[len(models):]:
        ax.set_visible(False)

    fig.suptitle("Temperatura por modelo", fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()
else:
    print("Sin datos de hardware para timelines")

In [ ]:
# Timelines de potencia
if not hw_ex.empty:
    models = sorted(ex["model_label"].unique())
    ncols = 2
    nrows = (len(models) + 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows))
    if len(models) == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, model in zip(axes, models):
        hw_m = hw_ex[hw_ex["model_label"] == model]
        if hw_m.empty:
            ax.text(0.5, 0.5, f"{model}\nSin datos", ha="center", va="center",
                   transform=ax.transAxes)
            ax.set_axis_off()
            continue

        sns.lineplot(data=hw_m, x="t_rel_s", y="internal_power_w", ax=ax, alpha=0.7)
        total_s = hw_m["t_rel_s"].max()
        ax.set_xlabel("Tiempo (min)" if total_s > 1800 else "Tiempo (s)")
        ax.set_ylabel("Potencia (W)")
        ax.set_title(f"{model}", fontweight="bold")
        ax.grid(True, alpha=0.3)

    for ax in axes[len(models):]:
        ax.set_visible(False)

    fig.suptitle("Potencia interna por modelo", fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()
else:
    print("Sin datos de hardware para potencia")

## Distribución CPU y Memoria

Boxplot y violin+strip de CPU durante inferencia y memoria total.

In [ ]:
# Distribución CPU y Memoria - Boxplot (FIX: rotation=45)
hw_ex = hw_full[hw_full["run_id"].isin(ex["run_id"])]

if not hw_ex.empty:
    hw_inference = hw_ex[hw_ex["cpu_usage_pct"] > 50]

    fig, axes = plt.subplots(2, 1, figsize=(12, 9))

    order = hw_ex.groupby("model_label")["cpu_usage_pct"].median().sort_values(ascending=False).index

    sns.boxplot(
        data=hw_inference, x="model_label", y="cpu_usage_pct",
        order=order, showfliers=False, ax=axes[0],
    )
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
    axes[0].set_title("Distribución de CPU durante inferencia")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("CPU (%)")
    axes[0].set_ylim(80, 102)

    sns.boxplot(
        data=hw_ex, x="model_label", y="mem_pct",
        order=order, showfliers=False, ax=axes[1],
    )
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
    axes[1].set_title("Distribución de memoria (%)")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Memoria (%)")

    fig.tight_layout()
    plt.show()
else:
    print("Sin datos de hardware")

In [ ]:
# Distribución CPU y Memoria - Violin + Strip (FIX: rotation=45)
if not hw_ex.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))

    active = hw_ex[hw_ex["cpu_usage_pct"] > 50]
    order_cpu = active.groupby("model_label")["cpu_usage_pct"].median().sort_values(ascending=False).index

    sns.violinplot(data=active, x="model_label", y="cpu_usage_pct",
                   order=order_cpu, inner=None, alpha=0.6, ax=axes[0])
    sns.stripplot(data=active, x="model_label", y="cpu_usage_pct",
                  order=order_cpu, size=2, alpha=0.3, color="black", ax=axes[0])
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
    axes[0].set_title("Distribución de CPU durante inferencia")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("CPU (%)")
    axes[0].set_ylim(80, 102)

    order_mem = hw_ex.groupby("model_label")["mem_pct"].median().sort_values(ascending=False).index
    sns.violinplot(data=hw_ex, x="model_label", y="mem_pct",
                   order=order_mem, inner=None, alpha=0.6, ax=axes[1])
    sns.stripplot(data=hw_ex, x="model_label", y="mem_pct",
                  order=order_mem, size=2, alpha=0.3, color="black", ax=axes[1])
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
    axes[1].set_title("Distribución de memoria (%)")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Memoria (%)")

    fig.tight_layout()
    plt.show()
else:
    print("Sin datos de hardware")

## Uso de swap por modelo

Swap timeline por modelo en small multiples.

In [ ]:
# Uso de swap por modelo
if not hw_ex.empty:
    models = sorted(ex["model_label"].unique())
    ncols = 2
    nrows = (len(models) + 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows))
    if len(models) == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, model in zip(axes, models):
        hw_m = hw_ex[hw_ex["model_label"] == model]
        if hw_m.empty:
            ax.text(0.5, 0.5, f"{model}\nSin datos", ha="center", va="center",
                   transform=ax.transAxes)
            ax.set_axis_off()
            continue

        sns.lineplot(data=hw_m, x="t_rel_s", y="swap_pct", ax=ax, alpha=0.7)
        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Swap (%)")
        ax.set_title(f"{model}", fontweight="bold")
        ax.grid(True, alpha=0.3)

    for ax in axes[len(models):]:
        ax.set_visible(False)

    fig.suptitle("Uso de swap por modelo", fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()
else:
    print("Sin datos de hardware")

## Análisis detallado de un run representativo

Visualizaciones profundas de temperatura, frecuencia, CPU/memoria para el run más representativo.

In [ ]:
# Find most representative run (longest duration or median metrics)
from monitorviz.viz import temp_freq_dual, cpu_memory_dual_phases, hw_distributions_panel

if not hw_ex.empty:
    # Select median run by duration
    run_durations = hw_ex.groupby("run_id")["t_rel_s"].max().sort_values()
    median_idx = len(run_durations) // 2
    representative_run_id = run_durations.iloc[median_idx]

    representative_run = None
    for r in coll.runs:
        if r.run_id == representative_run_id:
            representative_run = r
            break

    if representative_run:
        hw_r = hw_ex[hw_ex["run_id"] == representative_run_id]

        # Temp + freq
        try:
            fig = temp_freq_dual(hw_r, representative_run)
            plt.show()
        except Exception as e:
            print(f"No se pudo generar temp_freq_dual: {e}")

        # RAM + CPU with phases
        try:
            fig = cpu_memory_dual_phases(hw_r, representative_run)
            plt.show()
        except Exception as e:
            print(f"No se pudo generar cpu_memory_dual_phases: {e}")

        # Hardware distributions
        try:
            fig = hw_distributions_panel(hw_r, representative_run)
            plt.show()
        except Exception as e:
            print(f"No se pudo generar hw_distributions_panel: {e}")
    else:
        print("No se encontró run representativo")
else:
    print("Sin datos para análisis detallado")

## Métricas hardware comparativas por modelo

Barplot agrupado por modelo y configuración (ventilador).

In [ ]:
# Grouped bar chart hardware por modelo (FIX: rotation=45)
import re

def _plot_label(model_short: str, engine: str) -> str:
    """Abbreviate model name for axis labels."""
    name = model_short
    name = re.sub(r'-Q\d_K_[MS].*', '', name)
    name = re.sub(r':q\d_k_[ms]$', '', name)
    name = re.sub(r'-Instruct', '', name)
    name = re.sub(r'-2512', '', name)
    name = re.sub(r'-Distill-Qwen', '', name)
    name = re.sub(r'-3n-E2B-it', '', name)
    name = re.sub(r'-h-micro', '', name)
    name = re.sub(r'_', '.', name)
    name = name.strip('-')
    suffix = "O" if engine == "OLLAMA" else "L"
    return f"{name} ({suffix})"

hw_summary_rows = []
for _, row in ex.iterrows():
    hw_r = hw_full[hw_full["run_id"] == row["run_id"]]
    pm_r = pm_full[
        (pm_full["run_id"] == row["run_id"]) &
        (~pm_full["is_empty_generation"]) &
        (pm_full["latency_ms"] > 0)
    ]
    if hw_r.empty:
        continue
    active = hw_r[hw_r["cpu_usage_pct"] > 50]
    fan_label = "Con ventilador" if row.get("fan", False) else "Sin ventilador"
    hw_summary_rows.append({
        "model": _plot_label(row["model_short"], row["engine"]),
        "config": fan_label,
        "cpu_pct": active["cpu_usage_pct"].mean() if not active.empty else float("nan"),
        "tokens_s": pm_r["tokens_per_second"].mean() if not pm_r.empty else float("nan"),
        "power_w": hw_r["internal_power_w"].mean(),
        "mem_mb": (hw_r["mem_used_bytes"].mean() / (1024**2)) if not hw_r.empty else float("nan"),
    })

if hw_summary_rows:
    hw_summary = pd.DataFrame(hw_summary_rows)

    METRICS = [
        ("cpu_pct", "CPU media (%)", "(a)", (50, 105)),
        ("tokens_s", "Throughput (tok/s)", "(b)", None),
        ("power_w", "Potencia (W)", "(c)", None),
        ("mem_mb", "Memoria (MB)", "(d)", None),
    ]

    models = sorted(hw_summary["model"].unique())
    configs = sorted(hw_summary["config"].unique())
    colors = ["steelblue", "tomato"]
    x_pos = np.arange(len(models))
    n_cfg = len(configs)
    width = 0.75 / max(n_cfg, 1)

    fig, axes = plt.subplots(len(METRICS), 1, figsize=(12, 4*len(METRICS)))
    if len(METRICS) == 1:
        axes = [axes]

    for ax, (col, ylabel, label, ylim) in zip(axes, METRICS):
        data = hw_summary.dropna(subset=[col])
        for i, (cfg, color) in enumerate(zip(configs, colors)):
            vals = []
            for m in models:
                sub = data[(data["model"] == m) & (data["config"] == cfg)]
                vals.append(sub[col].mean() if not sub.empty else float("nan"))
            offset = (i - n_cfg / 2 + 0.5) * width
            ax.bar(x_pos + offset, vals, width * 0.9, label=cfg, color=color, alpha=0.85)

        ax.set_xticks(x_pos)
        ax.set_xticklabels(models, rotation=45, ha="right")  # FIX
        ax.set_ylabel(ylabel)
        if ylim:
            ax.set_ylim(*ylim)
        ax.text(-0.06, 1.03, label, transform=ax.transAxes,
                fontweight="bold", fontsize=12)
        if col == "cpu_pct":
            ax.legend(loc="upper right")

    fig.suptitle("Métricas hardware por modelo y configuración", y=1.01, fontsize=13)
    fig.tight_layout()
    plt.show()
else:
    print("Sin datos para grouped bar chart")

## Métricas de inferencia

Throughput, palabras/s, latencia, TTFT, perplejidad.

In [ ]:
# Helper function from nb03 - Adaptive visualization
def _plot_metric(
    df: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    unit_divisor: float = 1.0,
    unit_label: str = "",
    order: list[str] | None = None,
) -> None:
    """Adaptive visualization based on number of data points per model."""
    data = df.dropna(subset=[metric]).copy()
    data["_val"] = data[metric] / unit_divisor

    counts = data.groupby("model_label")["_val"].count()
    min_n = counts.min() if not counts.empty else 0

    _order = order or (
        data.groupby("model_label")["_val"]
        .mean().sort_values().index.tolist()
    )
    full_title = f"{title}" + (f" ({unit_label})" if unit_label else "")

    if min_n >= 10:
        # Histograms with KDE
        models = _order
        n_models = len(models)
        ncols = 2
        nrows = (n_models + 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4*nrows))
        if n_models == 1:
            axes = [axes]
        else:
            axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

        for ax, model in zip(axes, models):
            sub = data[data["model_label"] == model]["_val"]
            if not sub.empty:
                ax.hist(sub, bins=15, alpha=0.6, edgecolor="black", density=True)
                try:
                    from scipy.stats import gaussian_kde
                    kde = gaussian_kde(sub)
                    x_range = np.linspace(sub.min(), sub.max(), 100)
                    ax.plot(x_range, kde(x_range), 'r-', lw=2, label='KDE')
                except:
                    pass
                ax.set_title(model)
                ax.set_xlabel(unit_label if unit_label else metric)

        for ax in axes[n_models:]:
            ax.set_visible(False)

        fig.suptitle(full_title, fontsize=12, y=1.01)
        fig.tight_layout()

    elif min_n >= 3:
        # Stripplot with mean
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.stripplot(data=data, x="model_label", y="_val", order=_order, size=8, alpha=0.6, ax=ax)
        means = data.groupby("model_label")["_val"].mean().reindex(_order)
        ax.bar(range(len(means)), means, alpha=0.3, color="orange", width=0.3, label="Media")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_xlabel("")
        ax.set_ylabel(unit_label if unit_label else metric)
        ax.set_title(full_title)
        ax.legend()
        fig.tight_layout()

    else:
        # Horizontal barplot
        means = data.groupby("model_label")["_val"].agg(["mean", "std", "count"]).reindex(_order)
        fig, ax = plt.subplots(figsize=(10, max(6, len(means) * 0.4)))
        ax.barh(range(len(means)), means["mean"], xerr=means["std"], alpha=0.7, capsize=5)
        ax.set_yticks(range(len(means)))
        ax.set_yticklabels(means.index)
        ax.set_xlabel(unit_label if unit_label else metric)
        ax.set_title(full_title)

        for i, (idx, row) in enumerate(means.iterrows()):
            ax.text(row["mean"], i, f"  n={int(row['count'])}", va="center", fontsize=9)

        fig.tight_layout()

    plt.show()

# Metrics plots
pm_valid = pm_full[
    (pm_full["run_id"].isin(ex["run_id"])) &
    (~pm_full["is_empty_generation"]) &
    (pm_full["latency_ms"] > 0)
].copy()

if not pm_valid.empty:
    _plot_metric(pm_valid, "tokens_per_second",
                 ylabel="tokens/s", title="Throughput por modelo",
                 unit_label="tokens/s")

    _plot_metric(pm_valid, "words_per_second",
                 ylabel="palabras/s", title="Palabras por segundo por modelo",
                 unit_label="palabras/s")

    _plot_metric(pm_valid, "latency_ms",
                 ylabel="ms", title="Latencia total por prompt",
                 unit_divisor=1000, unit_label="s")

    _plot_metric(pm_valid, "time_to_first_token_ms",
                 ylabel="ms", title="TTFT por prompt",
                 unit_label="ms")
else:
    print("Sin datos de métricas de inferencia")

## ECDFs (Empirical Cumulative Distribution Functions)

ECDF de latencia y perplejidad por modelo.

In [ ]:
# ECDF de latencia
if not pm_valid.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.ecdfplot(data=pm_valid, x="latency_ms", hue="model_label", ax=ax)
    ax.set_title("ECDF de latencia")
    ax.set_xlabel("Latencia (ms)")
    ax.set_ylabel("Proporción acumulada")
    ax.legend(title="Modelo", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
    fig.tight_layout()
    plt.show()

    # ECDF de perplejidad
    ppl_valid = pm_valid.dropna(subset=["perplexity"])
    if not ppl_valid.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.ecdfplot(data=ppl_valid, x="perplexity", hue="model_label", ax=ax)
        ax.set_title("ECDF de perplejidad")
        ax.set_xlabel("Perplejidad")
        ax.set_ylabel("Proporción acumulada")
        ax.legend(title="Modelo", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
        fig.tight_layout()
        plt.show()
else:
    print("Sin datos para ECDFs")

## Distribución de longitud de respuesta

Histograma de eval_count (tokens generados) por modelo.

In [ ]:
# Response length distribution
if not pm_valid.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    models = sorted(pm_valid["model_label"].unique())
    for model in models:
        sub = pm_valid[pm_valid["model_label"] == model]
        axes[0].hist(sub["eval_count"], bins=20, alpha=0.5, label=model)

    axes[0].set_xlabel("Tokens generados (eval_count)")
    axes[0].set_ylabel("Frecuencia")
    axes[0].set_title("Distribución de longitud de respuesta por modelo")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    # Summary table
    summary_resp = pm_valid.groupby("model_label")["eval_count"].agg([
        ("count", "count"),
        ("mean", "mean"),
        ("std", "std"),
        ("min", "min"),
        ("max", "max"),
    ]).round(1).sort_values("mean", ascending=False)

    axes[1].axis("off")
    table_data = [["Modelo", "N", "Media", "Std", "Min", "Max"]]
    for idx, row in summary_resp.iterrows():
        table_data.append([
            idx,
            int(row["count"]),
            f"{row['mean']:.1f}",
            f"{row['std']:.1f}",
            f"{row['min']:.0f}",
            f"{row['max']:.0f}"
        ])

    ax_table = axes[1].table(cellText=table_data, loc="center", cellLoc="center")
    ax_table.auto_set_font_size(False)
    ax_table.set_fontsize(9)
    ax_table.scale(1, 1.5)

    fig.suptitle("Análisis de respuestas generadas", fontsize=12, y=1.01)
    fig.tight_layout()
    plt.show()
else:
    print("Sin datos para análisis de respuestas")

## Análisis específico de E0

Comparativa Fan vs No-fan, ranking de modelos.

In [ ]:
# Fan vs No-fan comparison
if "fan" in ex.columns:
    fan_comp = ex.pivot_table(
        index="model_label", columns="fan",
        values=["tokens_per_s_mean", "temp_max_c", "throttled_ratio"],
        aggfunc="mean"
    ).round(2)

    print("📊 Comparativa Fan vs No-fan:")
    display(fan_comp)

    # Ranking by throughput
    ranking = ex.groupby("model_label").agg({
        "tokens_per_s_mean": "mean",
        "ttft_ms_mean": "mean",
        "energy_per_token_j": "mean",
        "perplexity_geomean": "mean",
    }).round(2).sort_values("tokens_per_s_mean", ascending=False)

    print("\n🏆 Ranking de modelos (por throughput):")
    display(ranking)
else:
    print("Sin datos de fan")

## MBU: Utilización del ancho de banda

Memory Bandwidth Utilization.

In [ ]:
# MBU
if "mbu_pct" in ex.columns:
    mbu_data = ex.dropna(subset=["mbu_pct"])
    if not mbu_data.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        order = mbu_data.groupby("model_label")["mbu_pct"].mean().sort_values(ascending=False).index
        sns.barplot(data=mbu_data, x="model_label", y="mbu_pct", order=order, ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_ylabel("MBU (%)")
        ax.set_xlabel("")
        ax.set_title("Memory Bandwidth Utilization por modelo")

        # Add theoretical peak note
        ax.text(0.5, 0.95, "Nota: Cota superior teórica ~100%",
               transform=ax.transAxes, ha="center", fontsize=10,
               bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow"))

        fig.tight_layout()
        plt.show()
    else:
        print("Sin datos de MBU")
else:
    print("Sin columna de MBU")

## Trade-offs multidimensionales

Fronteras Pareto de varios pares de métricas.

In [ ]:
# Pareto fronts
try:
    from monitorviz.viz import pareto_panel_multi

    pareto_pairs = [
        ("perplexity_geomean", "tokens_per_s_mean",
         "Perplejidad", "Throughput (tok/s)", True, False),
        ("energy_per_token_j", "perplexity_geomean",
         "Energía/token (J)", "Perplejidad", True, True),
        ("power_mean_w", "tokens_per_s_mean",
         "Potencia (W)", "Throughput (tok/s)", True, False),
    ]

    required_cols = {c[0] for c in pareto_pairs} | {c[1] for c in pareto_pairs}
    available = required_cols.intersection(ex.columns)
    valid_pairs = [(p,q,pl,ql,*rest) for p,q,pl,ql,*rest in pareto_pairs if p in available and q in available]

    if valid_pairs:
        fig = pareto_panel_multi(ex, valid_pairs,
                                label_col="model_label",
                                title="E0 — Trade-offs multidimensionales (Frontera Pareto)")
        plt.show()
    else:
        print("Columnas insuficientes para Pareto")
except Exception as ex_err:
    print(f"Error en Pareto: {ex_err}")

## Desglose temporal de fases

Proporción de tiempo en prefill, decode, y overhead.

In [ ]:
# Phase breakdown
try:
    from monitorviz.viz import phase_breakdown_stacked

    if "phase_prompt_s" in pm_valid.columns and "phase_decode_s" in pm_valid.columns:
        fig = phase_breakdown_stacked(ex, normalized=True)
        plt.show()

        fig = phase_breakdown_stacked(ex, normalized=False)
        plt.show()
    else:
        print("Sin datos de fases")
except Exception as e:
    print(f"Error en phase_breakdown: {e}")

## Throttling heatmap

Matriz de throttling actividad por run y tiempo.

In [ ]:
# Throttling heatmap
try:
    from monitorviz.viz import throttling_heatmap

    runs_ex = [r for r in coll.runs if r.run_id in ex["run_id"].values]
    if hw_ex.empty or not runs_ex:
        print("Sin datos de hardware para heatmap de throttling")
    else:
        fig = throttling_heatmap(hw_full, runs_ex, bins=40,
                                title="E0 — Heatmap de throttling")
        plt.show()
except Exception as e:
    print(f"Error en throttling_heatmap: {e}")

## Eficiencia computacional y energética

Trabajo CPU, eficiencia, energía por token, relación potencia-CPU.

In [ ]:
# CPU work and efficiency
if "cpu_work_core_s" in ex.columns:
    _plot_metric(ex, "cpu_work_core_s",
                 ylabel="core·s", title="Trabajo CPU efectivo (core·s)",
                 unit_label="core·s")

    _plot_metric(ex, "cpu_efficiency",
                 ylabel="η", title="Eficiencia CPU",
                 unit_label="η")
else:
    print("Sin datos de eficiencia CPU")

# Energy per token
if "energy_per_token_j" in ex.columns:
    _plot_metric(ex, "energy_per_token_j",
                 ylabel="J", title="Energía por token",
                 unit_label="J")

# CPU work per token
if "cpu_work_core_s" in ex.columns and "tokens_per_s_mean" in ex.columns:
    cpu_per_token = ex.dropna(subset=["cpu_work_core_s", "tokens_per_s_mean"])
    if not cpu_per_token.empty:
        cpu_per_token["cpu_work_per_token"] = cpu_per_token["cpu_work_core_s"] / (cpu_per_token["tokens_per_s_mean"] + 1e-9)
        _plot_metric(cpu_per_token, "cpu_work_per_token",
                     ylabel="core·s/tok", title="Trabajo CPU por token",
                     unit_label="core·s/tok")

In [ ]:
# CPU vs Power relationship scatterplots
if "cpu_work_core_s" in ex.columns and "power_mean_w" in ex.columns:
    fig, ax = plt.subplots(figsize=(10, 6))

    valid_data = ex.dropna(subset=["cpu_work_core_s", "power_mean_w"])
    if not valid_data.empty:
        sns.scatterplot(data=valid_data, x="cpu_work_core_s", y="power_mean_w",
                       hue="model_label", s=100, ax=ax)
        ax.set_xlabel("Trabajo CPU (core·s)")
        ax.set_ylabel("Potencia media (W)")
        ax.set_title("Relación CPU - Potencia")
        ax.legend(fontsize=9)
        fig.tight_layout()
        plt.show()
else:
    print("Sin datos para scatterplot CPU-Potencia")

## Rendimiento vs arquitectura del modelo

TTFT vs embedding, throughput vs parámetros.

In [ ]:
# Model architecture analysis
arch_cols = ["n_params_from_info", "embedding_length_from_info"]
if all(col in ex.columns for col in arch_cols):
    arch_data = ex.dropna(subset=arch_cols).copy()
    if not arch_data.empty:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # TTFT vs embedding
        sns.scatterplot(data=arch_data, x="embedding_length_from_info", y="ttft_ms_mean",
                       hue="model_label", s=150, ax=axes[0])
        axes[0].set_xlabel("Embedding length")
        axes[0].set_ylabel("TTFT (ms)")
        axes[0].set_title("TTFT vs Embedding")
        axes[0].legend(fontsize=8)

        # tokens/s vs n_params
        arch_data["n_params_b"] = arch_data["n_params_from_info"] / 1e9
        sns.scatterplot(data=arch_data, x="n_params_b", y="tokens_per_s_mean",
                       hue="model_label", s=150, ax=axes[1])
        axes[1].set_xlabel("Parámetros (B)")
        axes[1].set_ylabel("tokens/s")
        axes[1].set_title("Throughput vs Parámetros")
        axes[1].legend(fontsize=8)

        # Model size vs n_params
        sns.scatterplot(data=arch_data, x="n_params_b", y="model_size_gb",
                       hue="model_label", s=150, ax=axes[2])
        axes[2].set_xlabel("Parámetros (B)")
        axes[2].set_ylabel("Tamaño (GB)")
        axes[2].set_title("Tamaño Modelo")
        axes[2].legend(fontsize=8)

        fig.tight_layout()
        plt.show()
    else:
        print("Sin datos de arquitectura")
else:
    print(f"Sin datos de arquitectura (falta {[c for c in arch_cols if c not in ex.columns]})")

## Diagramas polares comparativos

Vistas de rendimiento y hardware en coordenadas polares.

In [ ]:
# Radar charts helper
def _radar_chart(df, metrics, title, ax, colors):
    """Radar chart for E0."""
    cols = [m[0] for m in metrics]
    data = df.dropna(subset=cols).copy()
    if data.empty:
        ax.text(0.5, 0.5, "Sin datos", ha="center", va="center",
               transform=ax.transAxes)
        return

    labels = [m[1] for m in metrics]
    N = len(metrics)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    ax.set_theta_offset(np.pi/2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], size=7)
    ax.grid(color="gray", alpha=0.3)
    ax.set_title(title, pad=15, fontweight="bold")

    for (_, row), color in zip(data.iterrows(), colors[:len(data)]):
        vals = []
        for col, _, higher_is_better in metrics:
            col_min = data[col].min()
            col_max = data[col].max()
            rng = col_max - col_min if col_max != col_min else 1.0
            if higher_is_better:
                vals.append((data.loc[row.name, col] - col_min) / rng)
            else:
                vals.append((col_max - data.loc[row.name, col]) / rng)
        vals += vals[:1]
        ax.plot(angles, vals, lw=2, color=color, label=row["model_label"])
        ax.fill(angles, vals, alpha=0.12, color=color)

    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)

perf_metrics = [
    ("tokens_per_s_mean", "Throughput (tok/s)", True),
    ("ttft_ms_mean", "TTFT (ms)", False),
    ("perplexity_geomean", "Perplejidad", False),
    ("energy_per_token_j", "Energía/token", False),
]

hw_metrics = [
    ("temp_max_c", "Temp máx (°C)", False),
    ("power_mean_w", "Potencia (W)", False),
    ("throttled_ratio", "Throttling %", False),
]

fig = plt.figure(figsize=(16, 7))
colors_radar = ["steelblue", "tomato", "seagreen", "goldenrod", "mediumpurple"]

if all(m[0] in ex.columns for m in perf_metrics):
    ax1 = fig.add_subplot(121, polar=True)
    _radar_chart(ex, perf_metrics, "(a) Rendimiento", ax1, colors_radar)

if all(m[0] in ex.columns for m in hw_metrics):
    ax2 = fig.add_subplot(122, polar=True)
    _radar_chart(ex, hw_metrics, "(b) Hardware", ax2, colors_radar)

fig.suptitle("E0 — Diagramas polares (exterior = mejor)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Matriz de correlación Pearson

Correlaciones entre métricas de rendimiento y hardware.

In [ ]:
# Correlation heatmap
try:
    from monitorviz.viz import correlation_heatmap

    if len(ex) >= 3:
        fig = correlation_heatmap(ex,
                                 title=f"E0 — Correlación Pearson (n={len(ex)} runs)")
        plt.show()
    else:
        print(f"Datos insuficientes: solo {len(ex)} run(s), se necesitan ≥3.")
except Exception as e:
    print(f"Error en correlación: {e}")

## Alertas y anomalías detectadas

Runs con comportamientos anómalos o preocupantes.

In [ ]:
# Alerts and anomalies
alerts = []

# Throttling activo
throttled = ex[ex["throttled_ratio"] > 0.5]
if not throttled.empty:
    for _, row in throttled.iterrows():
        alerts.append(
            f"⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling"
        )

# Generaciones vacías
empty = ex[ex["n_empty_generations"] > 0]
if not empty.empty:
    for _, row in empty.iterrows():
        alerts.append(
            f"ℹ️  {row['model_label']}: {int(row['n_empty_generations'])} generaciones vacías"
        )

# Outliers en throughput (>2 std)
if "tokens_per_s_mean" in ex.columns:
    mean_tps = ex["tokens_per_s_mean"].mean()
    std_tps = ex["tokens_per_s_mean"].std()
    outliers_tps = ex[
        (ex["tokens_per_s_mean"] > mean_tps + 2*std_tps) |
        (ex["tokens_per_s_mean"] < mean_tps - 2*std_tps)
    ]
    if not outliers_tps.empty:
        for _, row in outliers_tps.iterrows():
            alerts.append(
                f"📊 {row['model_label']}: throughput outlier ({row['tokens_per_s_mean']:.2f} tok/s)"
            )

if alerts:
    for a in alerts:
        print(a)
else:
    print("✅ Sin alertas detectadas en E0")

## Conclusiones y discusión

### Confirmación/refutación de hipótesis

> TODO: Actualizar con resultados.

### Hallazgos principales

> TODO: Lista de descubrimientos.

### Limitaciones

> TODO: Limitaciones metodológicas.

### Implicaciones para el TFG

> TODO: Cómo contribuye este experimento a las conclusiones finales.

### Cuestiones abiertas

> TODO: Preguntas para trabajos futuros.